In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DB_PATH = Path("../data/rfid_events.db")

In [2]:
conn = sqlite3.connect(DB_PATH)

sessions_df  = pd.read_sql("SELECT * FROM sessions",  conn)
events_df    = pd.read_sql("SELECT * FROM events",    conn)
anomalies_df = pd.read_sql("SELECT * FROM anomalies", conn)
raw_reads_df = pd.read_sql("SELECT * FROM raw_reads", conn)

conn.close()


In [4]:
def compute_peak_entry_rssi(raw_rows: pd.DataFrame, entry_t0) -> float:
    """
    Returns the peak (maximum) RSSI of In reads within 5 seconds
    of the confirmed entry t0.
 
    Using raw RSSI float values directly rather than the categorical
    entry_rssi_strength label ("strong"/"moderate"/"weak") stored in
    the sessions table. The raw float is continuous and more informative
    — going via the category loses precision.
 
    Returns -100.0 if no In reads exist around the entry window.
    """
    if entry_t0 is None:
        return -100.0
    window    = abs(raw_rows["t0"] - entry_t0) <= 5
    in_reads  = raw_rows[window & (raw_rows["direction"] == "In")]["rssi"]
    return float(in_reads.max()) if not in_reads.empty else -100.0

In [5]:
def compute_rssi_slopes(raw_rows: pd.DataFrame, entry_t0) -> tuple:
    """
    Computes RSSI slope during the entry window for both directions.
 
    in_slope:
        How the signal evolved on the inside reader as the person crossed.
        Positive = signal getting stronger = person genuinely approaching.
        Negative = signal weakening = person moving away (suspicious entry).
 
    out_slope:
        How the outside reader behaved during the same window.
        If the Out signal is also rising while the person is entering,
        the outside reader was simultaneously picking up the tag —
        a strong indicator of a noisy environment or ghost read risk.
        If falling — the person genuinely moved away from the outside.
 
    Both slopes are computed as:
        (last_rssi - first_rssi) / number_of_reads
    This gives a per-read rate of change rather than absolute difference,
    making it comparable across sessions with different burst lengths.
 
    Returns (0.0, 0.0) if fewer than 2 reads exist in the window.
    """
    if entry_t0 is None:
        return 0.0, 0.0
 
    window    = abs(raw_rows["t0"] - entry_t0) <= 5
    in_rssi   = raw_rows[window & (raw_rows["direction"] == "In")]["rssi"].values
    out_rssi  = raw_rows[window & (raw_rows["direction"] == "Out")]["rssi"].values
 
    in_slope  = float((in_rssi[-1]  - in_rssi[0])  / len(in_rssi))  if len(in_rssi)  >= 2 else 0.0
    out_slope = float((out_rssi[-1] - out_rssi[0]) / len(out_rssi)) if len(out_rssi) >= 2 else 0.0
 
    return in_slope, out_slope

In [6]:
def compute_read_density(raw_rows: pd.DataFrame) -> float:
    """
    Computes reads per second across the full session duration.
 
    High density: person standing still near a reader (tag scanned
    many times per second). Low density: person moving quickly or
    staying far from readers. Very high density could indicate a tag
    resting directly on a reader antenna (suspicious).
 
    Only available from raw_reads — the events table has no equivalent.
 
    Returns 0.0 if the session has no rows.
    """
    if raw_rows.empty:
        return 0.0
    duration = raw_rows["t0"].max() - raw_rows["t0"].min()
    if duration == 0:
        return float(len(raw_rows))
    return round(len(raw_rows) / duration, 3)

In [7]:
def compute_antenna_count(raw_rows: pd.DataFrame) -> int:
    """
    Counts distinct antenna IDs that detected the tag during the session.
 
    Normal crossings are picked up by multiple antennas on the same
    reader as the person moves through its field.
    Single-antenna detection is unusual — it may indicate an odd angle,
    unusual distance, or a partially malfunctioning reader.
 
    Only available from raw_reads.
    """
    return int(raw_rows["antenna"].nunique())
 

In [10]:
rows = []
 
for _, session in sessions_df.iterrows():
    sid      = session["session_id"]
    raw_rows = raw_reads_df[raw_reads_df["session_id"] == sid].copy()
    evts     = events_df[events_df["session_id"] == sid]
 
    # Confirmed entry t0 — needed for signal window computation
    entry_events = evts[evts["event_type"] == "ENTRY"]
    entry_t0     = int(entry_events["t0"].iloc[0]) if not entry_events.empty else None
 
    # Dwell time: -1 for sessions with no clean exit (SESSION_ENDED_INSIDE
    # or EXIT_WITHOUT_ENTRY). Flags these as incomplete rather than zero.
    exit_events = evts[
        (evts["event_type"] == "EXIT") &
        (evts["dwell_time"].notna())
    ]
    dwell = float(exit_events["dwell_time"].iloc[0]) if not exit_events.empty else -1.0
 
    # Compute slope for both directions
    in_slope, out_slope = compute_rssi_slopes(raw_rows, entry_t0)
 
    rows.append({
        "session_id": sid,
 
        # ── From sessions / events table ──────────────────────────────
        "dwell_time":        dwell,
        "total_transitions": float(session["total_transitions"] or 0),
        "ghost_read_ratio":  float(session["ghost_read_ratio"]  or 0),
        "has_anomaly": float(len(anomalies_df[anomalies_df["session_id"] == sid]) > 0),
 
        # ── From raw_reads table ──────────────────────────────────────
        # These features are only computable because we stored raw reads.
        # The events table tells you WHAT happened.
        # The raw_reads table tells you HOW it happened — the signal
        # profile leading up to each event.
        "peak_entry_rssi":  compute_peak_entry_rssi(raw_rows, entry_t0),
        "rssi_slope_in":    in_slope,
        "rssi_slope_out":   out_slope,
        "read_density":     compute_read_density(raw_rows),
        "antenna_count":    float(compute_antenna_count(raw_rows)),
    })
 
features_df = pd.DataFrame(rows)
features_df

,session_id,dwell_time,total_transitions,ghost_read_ratio,has_anomaly,peak_entry_rssi,rssi_slope_in,rssi_slope_out,read_density,antenna_count
0,16062021_1_9,36.0,2.0,0.107,0.0,-65.0,0.000000,-0.400000,1.143,2.0
1,16062021_1_10,35.0,2.0,0.044,1.0,-64.0,-0.050633,0.428571,4.241,2.0
2,16062021_1_11,37.0,1.0,0.035,0.0,-57.0,0.000000,-0.214286,16.222,2.0
3,1667936153880_1,78.0,4.0,0.508,0.0,-60.5,-0.145833,1.500000,2.301,3.0
4,1667936153880_2,66.0,2.0,0.443,0.0,-58.0,-0.750000,0.461538,1.541,3.0
5,1666806605275,-1.0,2.0,0.022,1.0,-70.0,-0.375000,-0.250000,1.533,3.0
6,1666806815845,-1.0,0.0,0.158,1.0,-66.5,0.642857,-0.900000,0.792,3.0


In [11]:
FEATURE_COLS = [
    "dwell_time",
    "total_transitions",
    "ghost_read_ratio",
    "has_anomaly",
    "peak_entry_rssi",
    "rssi_slope_in",
    "rssi_slope_out",
    "read_density",
    "antenna_count",
]

In [13]:
from sklearn.preprocessing import StandardScaler

X = features_df[FEATURE_COLS].copy()

# StandardScaler centers each feature to mean=0 and std=1.
# Without this, dwell_time (range 35-78) would dominate over
# ghost_read_ratio (range 0-0.5) simply because of scale difference.
# Isolation Forest uses random feature splits — unscaled features
# bias the splits toward high-magnitude features.

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Original scale (mean, std):")
for col, mean, std in zip(FEATURE_COLS, scaler.mean_, scaler.scale_):
    print(f"  {col:<22} mean={mean:>8.3f}  std={std:>7.3f}")

Original scale (mean, std):
  dwell_time             mean=  35.714  std= 27.783
  total_transitions      mean=   1.857  std=  1.125
  ghost_read_ratio       mean=   0.188  std=  0.188
  has_anomaly            mean=   0.429  std=  0.495
  peak_entry_rssi        mean= -63.000  std=  4.367
  rssi_slope_in          mean=  -0.097  std=  0.392
  rssi_slope_out         mean=   0.089  std=  0.724
  read_density           mean=   3.968  std=  5.112
  antenna_count          mean=   2.571  std=  0.495


In [14]:
from sklearn.ensemble import IsolationForest

# contamination=0.35: we expect roughly 35% of sessions to be anomalous
# (3 out of 7 have rule-based anomalies = 43%, so 0.35 is conservative).
# This only affects the binary prediction threshold — not the raw scores.
# We focus on the continuous scores, not the binary labels.
#
# random_state=42: reproducible results across runs.
# n_estimators=200: more trees = more stable scores on small datasets.

model = IsolationForest(
    n_estimators=200,
    contamination=0.35,
    random_state=42,
)
model.fit(X_scaled)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",200
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.35
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [15]:
# decision_function returns raw scores — LOWER means MORE anomalous.
# We invert and normalize to [0, 1] so HIGHER = MORE anomalous.
# This makes the scores intuitive: 1.0 = most anomalous.

raw_scores  = model.decision_function(X_scaled)
epsilon     = 1e-9  # avoid division by zero
normalized  = 1 - (raw_scores - raw_scores.min()) / \
              (raw_scores.max() - raw_scores.min() + epsilon)

results_df = features_df[["session_id", "has_anomaly"]].copy()
results_df["ml_score"]      = normalized.round(4)
results_df["ml_prediction"] = model.predict(X_scaled)  # -1=anomaly, 1=normal
results_df["ml_flag"]       = results_df["ml_prediction"].map({-1: "anomaly", 1: "normal"})

results_df = results_df.sort_values("ml_score", ascending=False)
print("Anomaly scores (higher = more anomalous):")
display(results_df[["session_id", "has_anomaly", "ml_score", "ml_flag"]])

Anomaly scores (higher = more anomalous):


,session_id,has_anomaly,ml_score,ml_flag
6,1666806815845,1.0,1.0000,anomaly
3,1667936153880_1,0.0,0.9082,anomaly
2,16062021_1_11,0.0,0.7670,anomaly
4,1667936153880_2,0.0,0.6312,normal
5,1666806605275,1.0,0.4553,normal
1,16062021_1_10,1.0,0.3312,normal
0,16062021_1_9,0.0,0.0000,normal
